# Advanced Application Exercise: Physical AI Integration Challenge

## Overview
In this advanced exercise, you will integrate all concepts learned throughout the textbook to create a complete Physical AI system that can perceive its environment, make decisions, and execute actions safely. This exercise combines perception, cognition, control, and human-robot interaction principles.

## Learning Objectives
- Integrate perception, cognition, and control systems
- Implement safe human-robot interaction protocols
- Apply system integration patterns
- Evaluate performance of integrated systems

## Prerequisites
Before starting this exercise, ensure you have completed:
- All previous modules (1-4)
- Understanding of ROS 2 concepts
- Experience with simulation environments
- Basic knowledge of AI/ML for robotics

## Problem Statement

You are tasked with creating a Physical AI system for a home assistance scenario. The robot must:

1. Navigate to a target location while avoiding obstacles
2. Identify and pick up a specific object
3. Deliver the object to a designated drop-off location
4. Interact safely with humans in the environment
5. Handle unexpected situations gracefully

### Environment Setup
We'll use a simulated home environment with multiple rooms, furniture, and dynamic obstacles.

In [ ]:
# Import required libraries
import rospy
import numpy as np
import cv2
from sensor_msgs.msg import Image, LaserScan
from geometry_msgs.msg import Twist, Pose, Point
from nav_msgs.msg import Odometry
from tf.transformations import euler_from_quaternion
import tf
import actionlib
from move_base_msgs.msg import MoveBaseAction, MoveBaseGoal
from visualization_msgs.msg import Marker
import time

print("Required libraries imported successfully")

## Step 1: Environment Perception System

Implement a perception system that combines camera data, LIDAR data, and odometry to build a comprehensive understanding of the environment.

In [ ]:
class EnvironmentPerception:
    def __init__(self):
        # Initialize subscribers
        self.image_sub = rospy.Subscriber('/camera/rgb/image_raw', Image, self.image_callback)
        self.laser_sub = rospy.Subscriber('/scan', LaserScan, self.laser_callback)
        self.odom_sub = rospy.Subscriber('/odom', Odometry, self.odom_callback)
        
        # Initialize publishers
        self.marker_pub = rospy.Publisher('/visualization_marker', Marker, queue_size=10)
        
        # State variables
        self.current_pose = None
        self.obstacles = []
        self.target_object = None
        self.humans = []
        
        # Camera and sensor parameters
        self.camera_matrix = np.array([[500, 0, 320], [0, 500, 240], [0, 0, 1]])
        
    def image_callback(self, data):
        # Convert ROS Image message to OpenCV image
        # This is a simplified version - in practice you'd use cv_bridge
        pass
    
    def laser_callback(self, data):
        # Process LIDAR data to detect obstacles
        ranges = np.array(data.ranges)
        # Filter out invalid ranges
        valid_ranges = ranges[(ranges > data.range_min) & (ranges < data.range_max)]
        
        # Simple obstacle detection
        self.obstacles = []
        for i, range_val in enumerate(valid_ranges):
            if range_val < 1.0:  # Obstacle within 1 meter
                angle = data.angle_min + i * data.angle_increment
                x = range_val * np.cos(angle)
                y = range_val * np.sin(angle)
                self.obstacles.append((x, y))
    
    def odom_callback(self, data):
        # Extract robot pose from odometry
        self.current_pose = data.pose.pose
    
    def detect_objects(self, image):
        # Simple object detection using color thresholds
        # In practice, you'd use a deep learning model
        hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
        
        # Define color range for target object (red)
        lower_red = np.array([0, 100, 100])
        upper_red = np.array([10, 255, 255])
        mask1 = cv2.inRange(hsv, lower_red, upper_red)
        
        # Upper red range (HSV wraps around)
        lower_red = np.array([170, 100, 100])
        upper_red = np.array([180, 255, 255])
        mask2 = cv2.inRange(hsv, lower_red, upper_red)
        
        mask = mask1 + mask2
        
        # Find contours
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        target_objects = []
        for contour in contours:
            if cv2.contourArea(contour) > 100:  # Filter small contours
                # Calculate center of contour
                M = cv2.moments(contour)
                if M["m00"] != 0:
                    cx = int(M["m10"] / M["m00"])
                    cy = int(M["m01"] / M["m00"])
                    target_objects.append((cx, cy))
        
        return target_objects
    
    def get_environment_map(self):
        # Return a representation of the current environment
        env_map = {
            'robot_pose': self.current_pose,
            'obstacles': self.obstacles,
            'target_objects': self.target_object,
            'humans': self.humans
        }
        return env_map

# Initialize perception system
perception = EnvironmentPerception()
print("Environment perception system initialized")

## Step 2: Decision Making System

Implement a decision-making system that plans actions based on the perceived environment and task requirements.

In [ ]:
class DecisionMakingSystem:
    def __init__(self):
        self.action_client = actionlib.SimpleActionClient('move_base', MoveBaseAction)
        self.action_client.wait_for_server()
        
        # Task state
        self.task_state = 'SEARCHING'  # SEARCHING, NAVIGATING, PICKING_UP, DELIVERING, COMPLETE
        self.target_location = None
        self.dropoff_location = None
        self.object_picked = False
        
    def set_task(self, target_location, dropoff_location):
        self.target_location = target_location
        self.dropoff_location = dropoff_location
        self.task_state = 'SEARCHING'
        
    def make_decision(self, env_map):
        """Make decisions based on environment map and task state"""
        
        if self.task_state == 'SEARCHING':
            return self._search_for_object(env_map)
        elif self.task_state == 'NAVIGATING':
            return self._navigate_to_object(env_map)
        elif self.task_state == 'PICKING_UP':
            return self._pick_up_object(env_map)
        elif self.task_state == 'DELIVERING':
            return self._deliver_object(env_map)
        
    def _search_for_object(self, env_map):
        # Simple search behavior - move to target location
        if env_map['target_objects']:
            # Object detected, move to navigating state
            self.task_state = 'NAVIGATING'
            return self._navigate_to_object(env_map)
        else:
            # Navigate to target location to search
            goal = MoveBaseGoal()
            goal.target_pose.header.frame_id = "map"
            goal.target_pose.header.stamp = rospy.Time.now()
            goal.target_pose.pose.position.x = self.target_location[0]
            goal.target_pose.pose.position.y = self.target_location[1]
            goal.target_pose.pose.orientation.w = 1.0
            
            self.action_client.send_goal(goal)
            return "Navigating to search area"
    
    def _navigate_to_object(self, env_map):
        # Navigate closer to detected object
        if env_map['target_objects']:
            # Calculate approach position near object
            obj_pos = env_map['target_objects'][0]  # Simplified
            approach_pos = (obj_pos[0] - 0.5, obj_pos[1])  # 0.5m away
            
            goal = MoveBaseGoal()
            goal.target_pose.header.frame_id = "map"
            goal.target_pose.header.stamp = rospy.Time.now()
            goal.target_pose.pose.position.x = approach_pos[0]
            goal.target_pose.pose.position.y = approach_pos[1]
            goal.target_pose.pose.orientation.w = 1.0
            
            self.action_client.send_goal(goal)
            self.task_state = 'PICKING_UP'
            return "Approaching object for pickup"
        else:
            return "Object not in view, need to search further"
    
    def _pick_up_object(self, env_map):
        # Simulate object pickup
        self.object_picked = True
        self.task_state = 'DELIVERING'
        print("Object picked up successfully")
        return "Delivering object to drop-off location"
    
    def _deliver_object(self, env_map):
        # Navigate to drop-off location
        goal = MoveBaseGoal()
        goal.target_pose.header.frame_id = "map"
        goal.target_pose.header.stamp = rospy.Time.now()
        goal.target_pose.pose.position.x = self.dropoff_location[0]
        goal.target_pose.pose.position.y = self.dropoff_location[1]
        goal.target_pose.pose.orientation.w = 1.0
        
        self.action_client.send_goal(goal)
        
        # Check if goal reached
        if self.action_client.get_state() == actionlib.GoalStatus.SUCCEEDED:
            self.task_state = 'COMPLETE'
            print("Object delivered successfully")
            return "Task completed successfully"
        
        return "Delivering object..."

# Initialize decision making system
decision_system = DecisionMakingSystem()
print("Decision making system initialized")

## Step 3: Control System

Implement a control system that executes the planned actions while ensuring safety.

In [ ]:
class ControlSystem:
    def __init__(self):
        self.cmd_vel_pub = rospy.Publisher('/cmd_vel', Twist, queue_size=10)
        self.safety_enabled = True
        
    def move_robot(self, linear_x=0.0, angular_z=0.0):
        """Send velocity commands to robot"""
        cmd = Twist()
        cmd.linear.x = linear_x
        cmd.angular.z = angular_z
        
        if self.safety_enabled:
            # Check for safety conditions
            cmd = self._apply_safety_constraints(cmd)
        
        self.cmd_vel_pub.publish(cmd)
        
    def _apply_safety_constraints(self, cmd):
        """Apply safety constraints to velocity commands"""
        # Limit maximum velocities
        max_linear = 0.5  # m/s
        max_angular = 0.5  # rad/s
        
        cmd.linear.x = max(min(cmd.linear.x, max_linear), -max_linear)
        cmd.angular.z = max(min(cmd.angular.z, max_angular), -max_angular)
        
        # Emergency stop if obstacles too close
        if hasattr(self, 'obstacle_distance') and self.obstacle_distance < 0.3:
            cmd.linear.x = 0.0
            cmd.angular.z = 0.0
            
        return cmd
    
    def stop_robot(self):
        """Stop the robot immediately"""
        cmd = Twist()
        cmd.linear.x = 0.0
        cmd.angular.z = 0.0
        self.cmd_vel_pub.publish(cmd)
    
    def execute_navigation(self, goal_pose):
        """Execute navigation to goal pose"""
        # This would typically use move_base or similar navigation stack
        print(f"Navigating to goal: {goal_pose}")
        
# Initialize control system
control_system = ControlSystem()
print("Control system initialized")

## Step 4: Human-Robot Interaction Safety

Implement safety protocols for human-robot interaction.

In [ ]:
class HumanRobotInteraction:
    def __init__(self):
        self.human_proximity_threshold = 2.0  # meters
        self.safe_zone_active = False
        
    def check_human_safety(self, env_map):
        """Check if humans are in proximity and adjust behavior"""
        if 'humans' in env_map and env_map['humans']:
            for human in env_map['humans']:
                # Calculate distance to human (simplified)
                if hasattr(env_map['robot_pose'], 'position'):
                    robot_pos = env_map['robot_pose'].position
                    dist_to_human = np.sqrt((robot_pos.x - human[0])**2 + (robot_pos.y - human[1])**2)
                    
                    if dist_to_human < self.human_proximity_threshold:
                        self.safe_zone_active = True
                        return True  # Human too close
        
        self.safe_zone_active = False
        return False
    
    def adjust_behavior_for_safety(self):
        """Adjust robot behavior when humans are nearby"""
        if self.safe_zone_active:
            print("Human detected nearby - reducing speed and increasing safety margins")
            # Implement safety behaviors
            return True
        return False

# Initialize HRI system
hri_system = HumanRobotInteraction()
print("Human-robot interaction safety system initialized")

## Step 5: Main Integration Loop

Combine all systems in a main integration loop.

In [ ]:
class PhysicalAISystem:
    def __init__(self):
        self.perception = EnvironmentPerception()
        self.decision = DecisionMakingSystem()
        self.control = ControlSystem()
        self.hri = HumanRobotInteraction()
        
        # Set up the task
        target_location = (5.0, 3.0)  # Target object location
        dropoff_location = (1.0, 1.0)  # Drop-off location
        self.decision.set_task(target_location, dropoff_location)
        
    def run(self):
        """Main execution loop"""
        rate = rospy.Rate(10)  # 10 Hz
        
        while not rospy.is_shutdown():
            # Get current environment state
            env_map = self.perception.get_environment_map()
            
            # Check human safety
            human_nearby = self.hri.check_human_safety(env_map)
            if human_nearby:
                self.hri.adjust_behavior_for_safety()
            
            # Make decisions based on environment
            decision_result = self.decision.make_decision(env_map)
            print(f"Decision: {decision_result}")
            
            # Check if task is complete
            if self.decision.task_state == 'COMPLETE':
                print("Task completed successfully!")
                break
            
            rate.sleep()

# Initialize and run the complete system
if __name__ == '__main__':
    rospy.init_node('physical_ai_integration')
    
    ai_system = PhysicalAISystem()
    print("Physical AI Integration System initialized")
    print("Starting integration exercise...")
    
    # Run the system
    ai_system.run()

## Exercise Tasks

Complete the following tasks to enhance the Physical AI system:

1. **Enhance Object Detection**: Improve the object detection algorithm to recognize multiple object types using a pre-trained model
2. **Implement Path Planning**: Add A* or Dijkstra's algorithm for more efficient path planning around obstacles
3. **Add Grasping Simulation**: Implement a simple grasping simulation for the pickup phase
4. **Improve Safety Protocols**: Add more sophisticated safety checks based on human behavior prediction
5. **Performance Evaluation**: Add metrics to evaluate the system's performance (time to completion, path efficiency, safety violations)

## Evaluation Criteria

Your implementation will be evaluated based on:
- **Completeness**: All required components are implemented
- **Integration**: Systems work together seamlessly
- **Safety**: Proper safety protocols are in place
- **Efficiency**: Task completed in reasonable time
- **Robustness**: System handles edge cases gracefully